<a href="https://colab.research.google.com/github/EstephanyReyes/procesos-estocasticos/blob/main/M%C3%89TODOuniformizaci%C3%B3n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Actividad 5: Método de Uniformización para una CMTC
Reyes Fuentes Estephany Carolina
## Introducción

El método de uniformización es una técnica que permite calcular la matriz de transición P(t) de una Cadena de Markov en Tiempo Continuo (CMTC) mediante una cadena en tiempo discreto.

La idea consiste en construir una matriz de transición discreta P̂ y expresar P(t) como una serie infinita ponderada por probabilidades Poisson. Esto permite aproximar la matriz de transición utilizando únicamente un número finito de términos.

#Ejercicio 3
### **Teorema (matriz P(t)): La matriz de probabilidad de transición $P(t) = [p_{i,j} (t)]$ está dada por: $$P(t)\approx \sum_{k=0}^{\infty} e^{-rt}\frac{(rt)^k}{k!}\hat P^k$$**


3. Este teorema permite aproximar P(t) usando los primeros M términos de la serie infinita. Se obtienen buenos resultados si se elije

$$M\approx \max\{rt+5\sqrt{rt},20\}$$

1. Use esta propuesta para calcular $P(0.5), P(1) y P(5)$ para la matriz R del ejercicio 1.

$$R =
\begin{pmatrix}
0& 2& 3& 0  \\
4& 0& 2& 0 \\
0& 2& 0& 2 \\
1& 0& 3& 0
\end{pmatrix}$$

2. ¿Se verifica la ecuación de Chapman-Kolmogorov $P(1) = P(0.5)P(0.5)$?


In [20]:
import numpy as np
import math

In [21]:
R = np.array([
    [0, 2, 3, 0],
    [4, 0, 2, 0],
    [0, 2, 0, 2],
    [1, 0, 3, 0]
], dtype=float)

r = 6
# Calcular r_i
ri = np.sum(R, axis=1)
# Calcular P gorrito
P_gorro = np.zeros((4,4))
for i in range(4):
    for j in range(4):
        if i == j:
            P_gorro[i,j] = 1 - ri[i]/r
        else:
            P_gorro[i,j] = R[i,j]/r

print(P_gorro)

[[0.16666667 0.33333333 0.5        0.        ]
 [0.66666667 0.         0.33333333 0.        ]
 [0.         0.33333333 0.33333333 0.33333333]
 [0.16666667 0.         0.5        0.33333333]]


In [22]:
#código para calular P(t)
def calcular_P(t):
    M = math.ceil(max(r*t + 5*math.sqrt(r*t), 20))

    P = np.zeros((4,4))
    potencia = np.eye(4)

    for k in range(M+1):
        c = math.exp(-r*t) * (r*t)**k / math.factorial(k)
        P = P + c * potencia
        potencia = potencia @ P_gorro

    return P, M

In [23]:
#caclular P(0.5), P(1) y P(5)
P05, M05 = calcular_P(0.5)
P1, M1 = calcular_P(1)
P5, M5 = calcular_P(5)

print("P(0.5), M =", M05)
print(P05)

print("P(1), M =", M1)
print(P1)

print("P(5), M =", M5)
print(P5)

P(0.5), M = 20
[[0.25060868 0.2169646  0.38665694 0.14576979]
 [0.25313484 0.23836098 0.37440924 0.13409493]
 [0.1691195  0.19361489 0.42030102 0.2169646 ]
 [0.15801748 0.15744464 0.39833179 0.28620609]]
P(1), M = 20
[[0.20615112 0.20390203 0.3987096  0.1912358 ]
 [0.20828421 0.2053407  0.39789917 0.18847446]
 [0.19675849 0.19837934 0.40095869 0.20390203]
 [0.19204622 0.19399715 0.40147094 0.21248423]]
P(5), M = 58
[[0.19999963 0.19999963 0.39999925 0.19999962]
 [0.19999963 0.19999963 0.39999925 0.19999962]
 [0.19999962 0.19999962 0.39999925 0.19999963]
 [0.19999962 0.19999962 0.39999925 0.19999963]]


In [24]:
#verificación con Chapman-Kolmogorov P(1) = P(0.5)P(0.5)
producto = P05 @ P05

print("P(0.5)P(0.5):")
print(producto)

print("Diferencia P(1) - P(0.5)P(0.5):")
print(P1 - producto)

P(0.5)P(0.5):
[[0.20615141 0.20390232 0.39871018 0.19123609]
 [0.20828451 0.20534099 0.39789976 0.18847475]
 [0.19675878 0.19837963 0.40095927 0.20390232]
 [0.19204651 0.19399744 0.40147152 0.21248452]]
Diferencia P(1) - P(0.5)P(0.5):
[[-2.91016682e-07 -2.91016682e-07 -5.82033364e-07 -2.91016681e-07]
 [-2.91016682e-07 -2.91016682e-07 -5.82033364e-07 -2.91016681e-07]
 [-2.91016682e-07 -2.91016682e-07 -5.82033364e-07 -2.91016682e-07]
 [-2.91016681e-07 -2.91016681e-07 -5.82033364e-07 -2.91016683e-07]]


$P(1)\approx P(0.5)P(0.5)$

La diferencia máxima fue aproximadamente: $5.8\times 10^{-7}$

Por lo tanto, sí se verifica la ecuación de Chapman-Kolmogorov, considerando el pequeño error numérico por la aproximación de la serie.

#Ejercicio 4
El teorema (Cotas de error para P(t)) se puede usar suponiendo que se desea calcular P(t)
con una tolerancia $\varepsilon$. Se selecciona un valor M tal que:
$$\sum_{k=M+1}^{\infty}
e^{-rt}\frac{(rt)^k}{k!}
\leq \varepsilon$$

Y se puede implementar de acuerdo al siguiente algoritmo de uniformización para P(t):

1. Dados $R, t$ y $0 <\varepsilon<1$
2. Calcular $$r=\max_i\{r_i\}$$ donde $$r_i=\sum_{j\neq i} r_{ij}$$

3. Construir la matriz $\hat P$
4. Inicializar: $A=\hat P$,

$$B=e^{-rt}I$$ $$c=e^{-rt}$$ $$\text{suma}=c$$ $$k=1$$
5. Mientras $$\text{suma}<1-\varepsilon$$ hacer: $$c=c\frac{rt}{k}$$ $$B=B+cA$$ $$A=A\hat P$$ $$\text{suma}=\text{suma}+c$$ $$k=k+1$$
6. La matriz $B$ obtenida es una aproximación de $P(t)$ con error menor que $\varepsilon$

Repita el ejercicio 3 aplicando este algoritmo con una tolerancia ε = 0.00001 (indique el valor correspondiente de M en cada caso). Compare los resultados.

In [25]:
#Algoritmo de uniformización para P(t):
R = np.array([
    [0, 2, 3, 0],
    [4, 0, 2, 0],
    [0, 2, 0, 2],
    [1, 0, 3, 0]
], dtype=float)

def metodo_uniformizacion(R, t, epsilon):
    n = len(R)

    # Calcular r_i
    ri = []
    for i in range(n):
        suma_fila = 0
        for j in range(n):
            suma_fila = suma_fila + R[i][j]
        ri.append(suma_fila)

    # Calcular r = máximo de los r_i
    r = max(ri)

    # Calcular P gorrito
    Pgorro = np.zeros((n,n))

    for i in range(n):
        for j in range(n):
            if i == j:
                Pgorro[i][j] = 1 - ri[i]/r
            else:
                Pgorro[i][j] = R[i][j]/r

    # Pseudocódigo propuesto
    A = Pgorro
    B = math.exp(-r*t) * np.eye(n)
    c = math.exp(-r*t)
    suma = c
    k = 1

    while suma < 1 - epsilon:
        c = c * (r*t) / k
        B = B + c*A
        A = A @ Pgorro
        suma = suma + c
        k = k + 1

    M = k - 1 #El valor M correpeonde al último término agregado a la suma
              #como el contador k incrementa en cada iteración al terminar el ciclo
              #se toma M=k-1
    return B, M

In [26]:
#Aplicado con epsilon=0.00001
epsilon = 0.00001

P05, M05 = metodo_uniformizacion(R, 0.5, epsilon)
P1, M1 = metodo_uniformizacion(R, 1, epsilon)
P5, M5 = metodo_uniformizacion(R, 5, epsilon)

print("P(0.5)")
print(P05)
print("M =", M05)

print("P(1)")
print(P1)
print("M =", M1)

print("P(5)")
print(P5)
print("M =", M5)

P(0.5)
[[0.250608   0.21696392 0.38665557 0.14576911]
 [0.25313416 0.2383603  0.37440788 0.13409425]
 [0.16911882 0.19361421 0.42029966 0.21696392]
 [0.1580168  0.15744396 0.39833043 0.28620541]]
M = 13
P(1)
[[0.20615038 0.20390128 0.39870811 0.19123506]
 [0.20828347 0.20533995 0.39789768 0.18847371]
 [0.19675775 0.19837859 0.4009572  0.20390128]
 [0.19204548 0.1939964  0.40146945 0.21248349]]
M = 19
P(5)
[[0.19999853 0.19999853 0.39999705 0.19999852]
 [0.19999853 0.19999853 0.39999705 0.19999852]
 [0.19999852 0.19999852 0.39999705 0.19999853]
 [0.19999852 0.19999852 0.39999705 0.19999853]]
M = 56


Los resultados obtenidos con el algoritmo de tolerancia $\varepsilon=0.00001$ son prácticamente iguales a los del ejercicio 3. La diferencia es muy pequeña porque en ambos métodos se aproxima la misma serie infinita. La ventaja del algoritmo del ejercicio 4 es que el valor de M no se elige con una fórmula aproximada, sino que se detiene cuando la cola de la distribución Poisson es menor que la tolerancia indicada.